In [0]:

# --- repo coordinates ---
OWNER = "lindasylvie6"
REPO = "databricks_airbnb"
BRANCH = "main" 
FOLDER = "datasets"
# --- repo path ---
REPO_PATH = f"dbfs:/mnt/{OWNER}/{REPO}/{BRANCH}/{FOLDER}"
# Read the data
FILES = ["hosts.csv", "listings.csv", "bookings.csv"]

LANDING = "/Volumes/airbnb_obs/bronze/landing"

def raw_url(fname):
     path = f"{FOLDER}/{fname}"
     return f"https://raw.githubusercontent.com/{OWNER}/{REPO}/{BRANCH}/{path}"

for f in FILES:
    print(raw_url(f))

In [0]:
import requests, datetime, json

records = [] # collect audit metadata for each file

for fname in FILES:
    url = raw_url(fname)
    resp = requests.get(url, timeout = 30)
    resp.raise_for_status()  #fail loudly if github returns an error

    dest = f"{LANDING}/{fname}"
    with open(dest, "wb") as fh:  #volumes are a normal filesystem path
        fh.write(resp.content)

    meta = {
        "source_file": fname,
        "source_url": url,
        "bytes": len(resp.content),
        "landed_path": dest,
        "fetched_at_utc": datetime.datetime.utcnow().isoformat(), "http_status": resp.status_code,
    }
    records.append(meta)
    print(f"landed {fname} ({meta['bytes']:,} bytes)")

# keep the audit record as JSON next to the data
with open(f"{LANDING}/_audit.json", "w") as fh:
    json.dump(records, fh, indent=2)
print("\nManifest:")
print(json.dumps(records, indent = 2)) 

In [0]:
# verify the files landed
display(dbutils.fs.ls(LANDING))